# 26 · Pig counting calibration on validation data (YOLO26n-seg)
**Runtime: GPU (T4 or better)** · No training · Publication-ready figures in English

**Scientific boundary:** thresholds are selected using *validation* images only. The test partition is never evaluated or tuned in this notebook. Prior test-set observations (Experiments 24–25) are descriptive and do not enter selection.

**Outputs:** one formatted Excel report, figures in PNG (600 dpi), PDF and SVG, JSON provenance, and an automatically downloaded ZIP. Files are stored in the established project folder.

**Instructions:** Runtime → Change runtime type → **T4 GPU**, then Run all. If the exact validation directory cannot be inferred, the notebook stops with clear diagnostics instead of silently using test.

**Dataset note:** The source dataset stores several dates under a directory named `test/images`; the authoritative `particion_fecha_manifest.json` determines membership. Validation consists of the dates 2023-06-06, 2023-06-08, 2023-09-01 and 2023-09-04.


In [ ]:
!pip -q install "ultralytics>=8.4.0,<9" pyyaml xlsxwriter
import os, json, hashlib, zipfile, platform, datetime, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml, torch
from ultralytics import YOLO
from google.colab import drive, files
from IPython.display import display
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU required: Colab > Runtime > Change runtime type > T4 GPU.'
print('GPU:',torch.cuda.get_device_name(0))
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Project folder not found: {ROOT}'
OUT=ROOT/'03_Resultados'/'CERDOS_26_Calibracion_Conteo_Validacion'
FIG=OUT/'Figures'; FIG.mkdir(parents=True,exist_ok=True)
print('Project:',ROOT,'\nOutput:',OUT)
SEEDS=(2026,2027,2028)
THRESHOLDS=np.round(np.arange(.05,.951,.05),2)
PRED_CONF_FLOOR=.001; NMS_IOU=.70; IMGSZ=640; MAX_DET=100
# Optional explicit path if automatic discovery reports ambiguity:
VALID_IMAGES_OVERRIDE=''  # directory containing validation images, NEVER test
VALID_LABELS_OVERRIDE=''  # YOLO polygon segmentation labels corresponding to validation
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.labelsize':12,'axes.titlesize':13,
                     'figure.dpi':120,'savefig.dpi':600,'axes.spines.top':False,'axes.spines.right':False})

## 1. Provenance and verified model weights
Source weights are referenced by the existing experiment 23 report. The cryptographic SHA-256 fingerprints must match; this prevents mixing different training runs.

In [ ]:
# El ZIP 23 es opcional: en Drive los resultados pueden estar descomprimidos.
report_candidates=list(ROOT.rglob('CERDOS_23_YOLO26_Evaluacion_Resultados.zip'))
REPORT23=None
if report_candidates:
    with zipfile.ZipFile(report_candidates[0]) as zz:
        PROTOCOL=json.loads(zz.read('CERDOS_23_Protocolo.json'))
        REPORT23=json.loads(zz.read('CERDOS_23_Informe_Evaluacion.json'))
else:
    PROTOCOL={'seeds':list(SEEDS),'weights_sha256':{
      '2026':'ca7d45d3ec5226ba5d0193fa1dabe49862c75bc667c7685698e5ef31594e72bb',
      '2027':'a754fe8989cfba5f51d7e2392296e2ca992b0105007c3d37be832c8a0dffca3e',
      '2028':'43ae5a9bf1cf2b20f2fec4b4bd6a99992fd948a7482790a55d5e0e6cdd54c06'}}
    print('ZIP 23 not present in Drive; using independently recorded SHA-256 fingerprints from experiment 23.')
assert tuple(PROTOCOL['seeds'])==SEEDS
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
WEIGHTS={}
for seed in SEEDS:
    original=Path(REPORT23['modelos'][str(seed)]['best_pt']) if REPORT23 else Path('/nonexistent')
    found=original if original.exists() else ROOT/'03_Resultados'/'06_Entrenamiento'/'YOLO26n_fecha_v1'/f'seed_{seed}'/'weights'/'best.pt'
    assert found.is_file(), f'Missing trained weights seed {seed}: {found}'
    digest=sha256(found)
    assert digest==PROTOCOL['weights_sha256'][str(seed)], f'Weight SHA256 mismatch for seed {seed}'
    WEIGHTS[seed]=found
    print(seed,'verified:',found)

## 2. Strict validation partition discovery
The code uses *only* paths explicitly named `val`, `valid`, or `validation`. It verifies segmentation labels and aborts if the directory suggests test. Confirm its printed validation image count before inference. The labels must contain one polygon instance per line.

In [ ]:
IMG_EXT={'.jpg','.jpeg','.png','.bmp','.webp'}
def collect_images(d):return sorted(p for p in Path(d).rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT)
# Prefer the project's explicit fixed split val.txt, not a guessed validation directory.
split_candidates=[p for p in ROOT.rglob('val.txt') if p.is_file()]
print('val.txt candidates:',[str(p) for p in split_candidates])
if VALID_IMAGES_OVERRIDE:
    imgs=collect_images(Path(VALID_IMAGES_OVERRIDE))
    assert imgs,'No images at VALID_IMAGES_OVERRIDE'
else:
    assert len(split_candidates)==1, ('Expected one authoritative val.txt. Found: '+str(split_candidates)+
        '. If necessary, provide VALID_IMAGES_OVERRIDE and VALID_LABELS_OVERRIDE.')
    split=split_candidates[0]
    lines=[x.strip() for x in split.read_text().splitlines() if x.strip() and not x.strip().startswith('#')]
    def resolve_image(line):
        raw=Path(line)
        candidates=[raw,split.parent/raw,ROOT/raw,Path('/content')/raw]
        if raw.is_absolute():
            for anchor in ('/01_Datasets/', '/02_Segmentacion/', '/03_Resultados/'):
                if anchor in line: candidates.append(ROOT/anchor.strip('/')/line.split(anchor,1)[1])
        # Many split manifests list paths from a former Drive mount.
        if 'MyDrive/' in line:
            candidates.append(Path('/content/drive/MyDrive')/line.split('MyDrive/',1)[1])
        for p in candidates:
            if p.is_file():return p
        return None
    resolved=[resolve_image(x) for x in lines]
    missing_paths=[line for line,found in zip(lines,resolved) if found is None]
    assert not missing_paths, ('Validation split images not resolved. First entries: '+str(missing_paths[:5])+' | Example val.txt line: '+str(lines[0])+
        '. Check dataset mount or set VALID_IMAGES_OVERRIDE to the exact folder.')
    imgs=resolved
    assert len(set(map(str,imgs)))==len(imgs),'Duplicated images in val.txt'
# Independently verify no image name overlap with the frozen test split.
test_lists=list(ROOT.rglob('test.txt'))
if test_lists:
    test_names={Path(x.strip()).name for p in test_lists for x in p.read_text().splitlines() if x.strip()}
    assert not (test_names & {p.name for p in imgs}), 'VALIDATION/TEST IMAGE OVERLAP DETECTED'
# Original FaroPigSeg stores images under test/images; the authoritative split is DATE-BASED.
# Never infer experimental membership from the source dataset directory name.
import re
manifest_path=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'/'particion_fecha_manifest.json'
assert manifest_path.is_file(), f'Missing authoritative date-split manifest: {manifest_path}'
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
assignments=manifest['asignaciones_fecha']
def extract_date(im):
    match=re.search(r'(20\d{2}-\d{2}-\d{2})',im.name)
    assert match, f'Cannot determine acquisition date: {im.name}'
    return match.group(1)
wrong=[(p.name,extract_date(p),assignments.get(extract_date(p))) for p in imgs
       if assignments.get(extract_date(p))!='val']
assert not wrong, f'Validation leakage according to authoritative manifest: {wrong[:8]}'
assert len(imgs)==len(lines), 'Split manifest has unresolvable or duplicated paths'
print('Validation dates:',sorted({extract_date(p) for p in imgs}))
print('Data storage folder names do NOT determine split membership.')
if VALID_LABELS_OVERRIDE:
    label_base=Path(VALID_LABELS_OVERRIDE)
    def lbl_of(im):return label_base/(im.stem+'.txt')
else:
    def lbl_of(im):
        parts=list(im.parts)
        if 'images' in parts:
            i=parts.index('images'); parts[i]='labels';p=Path(*parts).with_suffix('.txt')
            if p.is_file():return p
        return im.parent.parent/'labels'/(im.stem+'.txt')
missing=[str(lbl_of(p)) for p in imgs if not lbl_of(p).is_file()]
assert not missing, ('Missing segmentation labels. Examples: '+str(missing[:5])+
    '. Supply VALID_LABELS_OVERRIDE if labels live in a separate directory.')
def gt_count(p):
    lines=[line.strip().split() for line in lbl_of(p).read_text().splitlines() if line.strip()]
    for row in lines:
        assert len(row)>=7 and (len(row)-1)%2==0, f'Non-polygon YOLO label in {lbl_of(p)}'
    return len(lines)
GT=np.array([gt_count(x) for x in imgs],int)
assert GT.sum()>0
print('VALIDATION IMAGES:',len(imgs),'GROUND TRUTH INSTANCES:',GT.sum())
print('VALIDATION source:',str(split_candidates[0]) if not VALID_IMAGES_OVERRIDE else VALID_IMAGES_OVERRIDE)
print('Example label:',lbl_of(imgs[0]))
print('Review these two paths. They MUST belong exclusively to VALIDATION.')

## 3. GPU predictions and fixed counting threshold grid
Run inference at confidence 0.001 using the previously reported image size and non-maximum suppression configuration. Each threshold simply counts predictions with confidence above its value; **nothing is optimized on test**.

In [ ]:
rows=[]; raw_counts=[]
for seed in SEEDS:
    print('Inferring seed',seed)
    model=YOLO(str(WEIGHTS[seed]))
    preds=model.predict(source=[str(p) for p in imgs],device=0,imgsz=IMGSZ,conf=PRED_CONF_FLOOR,
                        iou=NMS_IOU,max_det=MAX_DET,retina_masks=True,stream=True,verbose=False,save=False)
    num=0
    for im,gt,pred in zip(imgs,GT,preds):
        confidences=pred.boxes.conf.detach().cpu().numpy() if pred.boxes is not None else np.array([])
        for t in THRESHOLDS:
            c=int((confidences>=t).sum())
            rows.append({'seed':seed,'image':im.name,'threshold':float(t),'reference_count':int(gt),'predicted_count':c,
                         'signed_error':c-int(gt),'absolute_error':abs(c-int(gt))})
        raw_counts.append({'seed':seed,'image':im.name,'prediction_count_at_conf_floor':len(confidences)})
        num+=1
    assert num==len(imgs),f'Seed {seed}: {num} outputs for {len(imgs)} images'
    del model
    torch.cuda.empty_cache()
data=pd.DataFrame(rows)
assert data.groupby(['seed','threshold']).size().eq(len(imgs)).all()
print('Validation count table:',data.shape)

## 4. Model selection: MAE and deterministic tie-breaking
For each seed, select the threshold with minimum validation MAE. Ties are resolved by the *lowest threshold* (predefined convention). The average of seed-specific thresholds is **not** automatically transferred to other models. These estimates are validation results and may be optimistic because selection and reporting use the same sample.

In [ ]:
summary=data.groupby(['seed','threshold'],as_index=False).agg(
    mae=('absolute_error','mean'),bias=('signed_error','mean'),
    rmse=('signed_error',lambda v: float(np.sqrt(np.mean(np.square(v))))),
    exact_count_fraction=('absolute_error',lambda v:float(np.mean(np.asarray(v)==0))),n_images=('image','size'))
selected=summary.sort_values(['seed','mae','threshold']).drop_duplicates('seed').reset_index(drop=True)
print('VALIDATION-SELECTED THRESHOLDS:')
display(selected)
# Thresholds above are fit ONLY on validation; results must not be described as independent test evidence.

## 5. Publication-ready English figures and Excel tables

In [ ]:
def export_figure(fig,stem):
    fig.tight_layout()
    for ext in ('png','pdf','svg'):fig.savefig(FIG/(stem+'.'+ext),dpi=600,bbox_inches='tight')
    plt.show();plt.close(fig)
COLORS=['#0072B2','#D55E00','#009E73']
fig,ax=plt.subplots(figsize=(8.4,5.2))
for col,seed in zip(COLORS,SEEDS):
    df=summary[summary.seed==seed]
    ax.plot(df.threshold,df.mae,label=f'Seed {seed}',color=col,marker='o',markersize=3.5,linewidth=2)
    s=selected[selected.seed==seed].iloc[0]
    ax.scatter([s.threshold],[s.mae],color=col,s=70,zorder=5,marker='D')
ax.set(xlabel='Confidence threshold',ylabel='Mean absolute counting error (pigs/image)',
       title='Validation-set counting sensitivity')
ax.legend(frameon=False);ax.grid(alpha=.2)
export_figure(fig,'Fig26_01_Validation_Count_MAE')
fig,ax=plt.subplots(figsize=(8.4,5.2))
for col,seed in zip(COLORS,SEEDS):
    df=summary[summary.seed==seed]
    ax.plot(df.threshold,df.bias,color=col,label=f'Seed {seed}',linewidth=2)
ax.axhline(0,color='black',linestyle='--',linewidth=1)
ax.set(xlabel='Confidence threshold',ylabel='Mean signed error (predicted − reference)',
       title='Validation-set counting bias')
ax.grid(alpha=.2);ax.legend(frameon=False)
export_figure(fig,'Fig26_02_Validation_Count_Bias')
# Excel with a readable layout and filters, all worksheet names and headings in English.
excel=OUT/'CERDOS_26_Validation_Count_Calibration.xlsx'
with pd.ExcelWriter(excel,engine='xlsxwriter') as writer:
    tables={'Selected_Validation':selected,'Threshold_Grid':summary,'Per_Image_Validation':data,
            'Prediction_Diagnostics':pd.DataFrame(raw_counts)}
    for tab,df in tables.items():
        df.to_excel(writer,sheet_name=tab,index=False)
        ws=writer.sheets[tab];ws.freeze_panes(1,0);ws.autofilter(0,0,len(df),len(df.columns)-1)
        for j,k in enumerate(df.columns):ws.set_column(j,j,min(38,max(15,len(k)+3)))
        fmt=writer.book.add_format({'bold':True,'bg_color':'#17365D','font_color':'white'})
        for j,k in enumerate(df.columns):ws.write(0,j,k,fmt)
print('Written:',excel)

## 6. Provenance, archive, and automatic download
The exported ZIP contains the data and publication-ready figures; no copies of images, model weights, or personal files are included.

In [ ]:
provenance={'experiment':26,'purpose':'VALIDATION ONLY: pig counting threshold selection',
 'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'device':torch.cuda.get_device_name(0),'torch':torch.__version__,
 'ultralytics_version':__import__('ultralytics').__version__,
 'source_experiment23':str(report_candidates[0]) if report_candidates else 'fingerprints from experiment 23',
 'source_experiment23_sha256':sha256(report_candidates[0]) if report_candidates else None,
 'validation_split':str(split_candidates[0]) if not VALID_IMAGES_OVERRIDE else VALID_IMAGES_OVERRIDE,
 'validation_label_example':str(lbl_of(imgs[0])),
 'validation_images':len(imgs),'validation_gt_instances':int(GT.sum()),
 'validation_image_names_sha256':hashlib.sha256('\n'.join(p.name for p in imgs).encode()).hexdigest(),
 'weights_sha256':{str(s):sha256(WEIGHTS[s]) for s in SEEDS},
 'inference':{'imgsz':IMGSZ,'conf':PRED_CONF_FLOOR,'iou':NMS_IOU,'max_det':MAX_DET,'retina_masks':True},
 'threshold_grid':[float(t) for t in THRESHOLDS],
 'selection_metric':'validation MAE, tie: smallest threshold',
 'selected':selected.to_dict('records'),
 'limitations':'No test evaluation; selected validation MAE is not an unbiased external estimate. For rigorous comparison use untouched holdout or new independent data.'}
meta=OUT/'CERDOS_26_Provenance.json';meta.write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
archive=OUT/'CERDOS_26_Resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [excel,meta,*sorted(FIG.glob('Fig26_*'))]:z.write(p,p.relative_to(OUT))
print('Saved in Google Drive:',OUT)
print('ZIP:',archive,'(',round(archive.stat().st_size/1024,1),'KB)')
files.download(str(archive))